# Validación RAGAS del pipeline RAG · Sprint 4

**Objetivo:** Medir la calidad del pipeline RAG (rag_service.py) contra el golden set usando el framework RAGAS.

**Métricas objetivo:**
- `faithfulness` ≥ **0.75** — fidelidad al contexto (no alucina)
- `answer_relevance` ≥ **0.70** — respuesta aborda la pregunta
- `context_precision` — señal: proporción de chunks recuperados que son relevantes
- `context_recall` — señal: proporción de ground truth cubierto por chunks

**Protocolo de escalamiento:** si alguna métrica < umbral tras 3 iteraciones, detener y consultar al tesista.

**Iteraciones a evaluar:**
1. Baseline actual: `chunk=500/overlap=50`, `threshold=0.70`, `top_k=5`
2. Alternativa v4.0: `chunk=semántico 15%`, `threshold=0.65`, `top_k=5`
3. Ajuste fino según resultados previos

## 1. Setup

In [ ]:
import os
import sys
import json
import asyncio
from pathlib import Path

# Asegurar que el backend sea importable
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
from datasets import Dataset

# RAGAS
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_ollama import ChatOllama, OllamaEmbeddings

# Backend (ajustar PYTHONPATH si necesario)
from app.config import settings
from app.database import async_session_maker
from app.services.rag_service import query_rag, _semantic_search
from app.services.embed_service import embed_query
import redis.asyncio as aioredis

## 2. Cargar golden set

In [ ]:
GOLDEN_SET_PATH = Path("../tests/fixtures/golden_set.json")

with open(GOLDEN_SET_PATH, encoding="utf-8") as f:
    golden = json.load(f)

print(f"Golden set version: {golden['meta']['version']}")
print(f"Total preguntas: {golden['meta']['total_questions']}")
print(f"Módulos: {golden['meta']['modules_covered']}")

df = pd.DataFrame(golden["questions"])
df[["id", "module", "type", "difficulty", "question"]].head(10)

## 3. Ejecutar pipeline RAG contra cada pregunta

Captura `answer` + `contexts` recuperados para alimentar RAGAS.

In [ ]:
async def run_rag_on_question(q_text: str):
    """Ejecuta embed → retrieve → LLM y devuelve answer + contexts."""
    redis_client = aioredis.from_url(settings.REDIS_URL, decode_responses=True)
    async with async_session_maker() as db:
        # 1. Recuperar contextos directamente
        query_vec = await embed_query(q_text)
        chunks = await _semantic_search(query_vec, db)
        contexts = [c["content"] for c in chunks]

        # 2. Ejecutar RAG completo (limpia caché antes para evitar hits de corridas previas)
        import hashlib
        h = hashlib.sha256(q_text.strip().lower().encode()).hexdigest()[:16]
        await redis_client.delete(f"rag:{h}")

        result = await query_rag(q_text, session_history=[], db=db, redis_client=redis_client)
    await redis_client.aclose()
    return result["content"], contexts

async def build_dataset():
    records = []
    for q in golden["questions"]:
        answer, contexts = await run_rag_on_question(q["question"])
        records.append({
            "question": q["question"],
            "answer": answer,
            "contexts": contexts,
            "ground_truth": q["ground_truth"],
            "reference": q["ground_truth"],  # alias RAGAS
            "module": q["module"],
            "type": q["type"],
            "difficulty": q["difficulty"],
        })
        print(f"✓ Q{q['id']:02d} ({q['module']}/{q['type']}): {q['question'][:60]}")
    return records

records = await build_dataset()
dataset = Dataset.from_list(records)
print(f"\nDataset listo: {len(dataset)} ejemplos")

## 4. Configurar RAGAS con Ollama (auto-hospedado)

Por defecto RAGAS usa OpenAI; lo sobreescribimos con Ollama local para respetar la restricción de no-APIs-pagas.

In [ ]:
llm = ChatOllama(
    base_url=settings.OLLAMA_BASE_URL,
    model=settings.OLLAMA_MODEL,
    temperature=0,
    num_ctx=4096,
)
embeddings = OllamaEmbeddings(
    base_url=settings.OLLAMA_BASE_URL,
    model=settings.OLLAMA_EMBED_MODEL,
)

ragas_llm = LangchainLLMWrapper(llm)
ragas_embeddings = LangchainEmbeddingsWrapper(embeddings)

metrics = [faithfulness, answer_relevancy, context_precision, context_recall]
for m in metrics:
    m.llm = ragas_llm
    if hasattr(m, "embeddings"):
        m.embeddings = ragas_embeddings

## 5. Ejecutar evaluación RAGAS

In [ ]:
results = evaluate(dataset, metrics=metrics, llm=ragas_llm, embeddings=ragas_embeddings)
print(results)
results_df = results.to_pandas()
results_df.head()

## 6. Análisis por módulo, tipo y dificultad

In [ ]:
metric_cols = ["faithfulness", "answer_relevancy", "context_precision", "context_recall"]

print("\n== Promedios globales ==")
print(results_df[metric_cols].mean().round(3))

print("\n== Por módulo ==")
print(results_df.groupby("module")[metric_cols].mean().round(3))

print("\n== Por tipo ==")
print(results_df.groupby("type")[metric_cols].mean().round(3))

print("\n== Por dificultad ==")
print(results_df.groupby("difficulty")[metric_cols].mean().round(3))

## 7. Verificar umbrales

In [ ]:
UMBRALES = {
    "faithfulness": 0.75,
    "answer_relevancy": 0.70,
}

promedios = results_df[metric_cols].mean()
print("Resultado vs umbrales:")
for metric, umbral in UMBRALES.items():
    valor = promedios[metric]
    status = "✅ OK" if valor >= umbral else "❌ BAJO"
    print(f"  {metric}: {valor:.3f} (umbral {umbral}) {status}")

## 8. Casos límite — peores respuestas

In [ ]:
worst = results_df.nsmallest(5, "faithfulness")[
    ["module", "type", "question", "faithfulness", "answer_relevancy"]
]
worst

## 9. Persistir resultados para el reporte

Snapshot etiquetado por iteración. Alimenta `docs/reporte-RAGAS.docx`.

In [ ]:
from datetime import datetime

ITERACION = "baseline"  # cambiar por cada corrida: baseline | v4_config | tuned_v1 ...

out_dir = Path("./ragas_runs")
out_dir.mkdir(exist_ok=True)

stamp = datetime.now().strftime("%Y%m%d_%H%M")
out_path = out_dir / f"{stamp}_{ITERACION}.csv"
results_df.to_csv(out_path, index=False)
print(f"Guardado: {out_path}")